In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=False,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
wildchat_dataset = ds_client.datasets[0]
wildchat_dataset.describe()

In [ ]:
import dotenv
from langchain.chat_models import init_chat_model
import os
import pandas as pd

dotenv.load_dotenv(wildchat_dataset.mock_path / "credentials.env")

llm = init_chat_model(
    model=os.environ["OPENROUTER_MODEL_NAME"],
    model_provider=os.environ["MODEL_PROVIDER"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=os.environ["OPENROUTER_API_URL"],
)

In [ ]:
mock_data = pd.read_parquet(
    wildchat_dataset.mock_path / "data.parquet",
)

mock_data

In [ ]:
from rds_chat_analysis.prompts import format_facet_extraction_prompt, FACET_EXTRACTORS

print("Available facet extractors:")
for key in FACET_EXTRACTORS.keys():
    print(f"- {key}")


conversation = mock_data.iloc[10]["conversation"]

messages = format_facet_extraction_prompt(
    conversation=list(conversation),
    **FACET_EXTRACTORS["Request"],
)

In [ ]:
from rich.console import Console
from langchain_core.messages import BaseMessage
import textwrap


def display_messages(messages: list[BaseMessage], width: int = 120):
    console = Console(highlight=False, soft_wrap=True)
    for msg in messages:
        role = msg.type.capitalize()
        wrapped = "\n".join(
            textwrap.fill(line, width=width)
            for line in msg.content.strip().splitlines()
        )
        console.print(f"[bold green]{role.upper()}:[/bold green]\n{wrapped}")


display_messages(messages)

In [ ]:
print(messages[-1])
result = llm.invoke(messages)
print(result)

In [ ]:
def extract_answer(text: str) -> str | None:
    text = text.strip()
    if "<answer>" in text:
        text = text.split("<answer>", 1)[-1]
    if "</answer>" in text:
        text = text.split("</answer>", 1)[0]
    return text.strip() or None


answer = extract_answer(result.content)

print(textwrap.fill(answer, width=120))

In [ ]:
print(result.model_dump_json(indent=2))